> **Production note (2026-06-21):** The script pipeline in `scripts/` is the source of truth for final outputs. This notebook is retained for exploration and narrative context; run the README pipeline for reproducible delivery artifacts.


# 10 · Master Dataset

Combines every dataset in `data/inputs/` into a single master table keyed on **Fecha × CCAA**.

**Run order**: after notebooks 02, 03 and before notebooks 04–09.

| Dataset | Granularity | Merge strategy |
|---|---|---|
| `consumo_biodiesel_ccaa.csv` | Monthly × CCAA | **Base table** |
| `consumo_biodiesel_targets.csv` | Monthly × CCAA (5 targets) | Left join on Fecha + CCAA → adds `Target` flag |
| `macro_indicadores_ine.csv` | Monthly, national | Left join on Fecha → broadcast to all CCAAs |
| `brent_oil_price_monthly_2023_onwards.csv` | Monthly, national | Left join on Fecha → broadcast to all CCAAs |
| `precios_combustibles_2023/24/25.csv` | Daily × Province × Product | Aggregate daily→monthly, Province→CCAA, pivot Product |

**Not merged** — see final section for details:
- `consumo_biodiesel_provincial.csv`
- `turismo_visitantes_ccaa.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT      = Path('..').resolve()
DATA_INPUTS    = REPO_ROOT / 'data' / 'inputs'
DATA_OUTPUTS   = REPO_ROOT / 'data' / 'outputs'

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

## 1 · Base table — biodiesel consumption by CCAA

In [ ]:
base = pd.read_csv(DATA_INPUTS / 'consumo_biodiesel_ccaa.csv')
base['Fecha'] = base['Fecha'].astype(str)
print(f'Base table: {base.shape}  — {base["CCAA"].nunique()} CCAAs × {base["Fecha"].nunique()} months')
base.head()

## 2 · Add `Target` flag from consumo_biodiesel_targets

In [ ]:
targets_df = pd.read_csv(DATA_INPUTS / 'consumo_biodiesel_targets.csv')
TARGET_CCAAS = set(targets_df['CCAA'].unique())

master = base.copy()
master['Target'] = master['CCAA'].isin(TARGET_CCAAS).astype(int)

print(f'After Target flag: {master.shape}')
print(f'CCAAs flagged as Target: {master[master["Target"]==1]["CCAA"].unique().tolist()}')

## 3 · Add macro indicators (INE) — national, broadcast to all CCAAs

In [ ]:
macro = pd.read_csv(DATA_INPUTS / 'macro_indicadores_ine.csv')
macro['Fecha'] = macro['Fecha'].astype(str)

master = master.merge(macro, on='Fecha', how='left')
print(f'After macro merge: {master.shape}')
print(f'Macro NaN rows: {master[["IPI_original","Tasa_paro"]].isna().sum().to_dict()}')

## 4 · Add Brent oil price — national, broadcast to all CCAAs

In [ ]:
brent = pd.read_csv(DATA_INPUTS / 'brent_oil_price_monthly_2023_onwards.csv',
                    usecols=['observation_date', 'brent_price_usd_per_barrel'])
brent['Fecha'] = pd.to_datetime(brent['observation_date']).dt.to_period('M').astype(str)
brent = brent.drop(columns='observation_date').rename(columns={'brent_price_usd_per_barrel': 'Precio_Brent_USD'})

master = master.merge(brent, on='Fecha', how='left')
print(f'After Brent merge: {master.shape}')
print(f'Brent NaN rows: {master["Precio_Brent_USD"].isna().sum()}')

## 5 · Add fuel prices — aggregate from daily × Province to monthly × CCAA

Steps:
1. Concat 2023/24/25 raw files
2. Aggregate daily → monthly mean per (Province, Product)
3. Pivot Product into columns (PAI + PVP for 4 product types)
4. Map Province → CCAA using official mapping
5. Aggregate provinces within each CCAA by simple mean
6. Merge to master on Fecha + CCAA
7. Fill ESPAÑA rows with national mean across all CCAAs

In [ ]:
PROVINCE_CCAA = {
    'Albacete':               'Castilla - La Mancha',
    'Alicante/Alacant':       'Comunitat Valenciana',
    'Almería':                'Andalucía',
    'Araba/Álava':            'País Vasco',
    'Asturias':               'Asturias, Principado de',
    'Badajoz':                'Extremadura',
    'Balears, Illes':         'Balears, Illes',
    'Barcelona':              'Cataluña',
    'Bizkaia':                'País Vasco',
    'Burgos':                 'Castilla y León',
    'Cantabria':              'Cantabria',
    'Castellón/Castelló':     'Comunitat Valenciana',
    'Ceuta':                  'Ceuta',
    'Ciudad Real':            'Castilla - La Mancha',
    'Coruña, A':              'Galicia',
    'Cuenca':                 'Castilla - La Mancha',
    'Cáceres':                'Extremadura',
    'Cádiz':                  'Andalucía',
    'Córdoba':                'Andalucía',
    'Gipuzkoa':               'País Vasco',
    'Girona':                 'Cataluña',
    'Granada':                'Andalucía',
    'Guadalajara':            'Castilla - La Mancha',
    'Huelva':                 'Andalucía',
    'Huesca':                 'Aragón',
    'Jaén':                   'Andalucía',
    'León':                   'Castilla y León',
    'Lleida':                 'Cataluña',
    'Lugo':                   'Galicia',
    'Madrid':                 'Madrid, Comunidad de',
    'Melilla':                'Melilla',
    'Murcia':                 'Murcia, Región de',
    'Málaga':                 'Andalucía',
    'Navarra':                'Navarra, Comunidad Foral de',
    'Ourense':                'Galicia',
    'Palencia':               'Castilla y León',
    'Palmas, Las':            'Canarias',
    'Pontevedra':             'Galicia',
    'Rioja, La':              'Rioja, La',
    'Salamanca':              'Castilla y León',
    'Santa Cruz de Tenerife': 'Canarias',
    'Segovia':                'Castilla y León',
    'Sevilla':                'Andalucía',
    'Soria':                  'Castilla y León',
    'Tarragona':              'Cataluña',
    'Teruel':                 'Aragón',
    'Toledo':                 'Castilla - La Mancha',
    'Valencia/València':      'Comunitat Valenciana',
    'Valladolid':             'Castilla y León',
    'Zamora':                 'Castilla y León',
    'Zaragoza':               'Aragón',
    'Ávila':                  'Castilla y León',
}

In [ ]:
raw_chunks = []
for yr in [2023, 2024, 2025]:
    df = pd.read_csv(
        DATA_INPUTS / f'precios_combustibles_{yr}.csv',
        sep=';', encoding='utf-8', encoding_errors='replace'
    )
    df.columns = [c.lstrip('\ufeff').strip() for c in df.columns]
    raw_chunks.append(df)

raw = pd.concat(raw_chunks, ignore_index=True)
raw.rename(columns={
    'Fecha Precio':                           'Fecha_dia',
    'Promedio de Pai Diario CUBO \u20ac/litro': 'PAI',
    'Promedio de Pvp Diario CUBO \u20ac/litro': 'PVP',
}, inplace=True)

raw['Fecha'] = pd.to_datetime(raw['Fecha_dia']).dt.to_period('M').astype(str)

for col in ['PAI', 'PVP']:
    raw[col] = pd.to_numeric(
        raw[col].astype(str).str.replace(',', '.', regex=False), errors='coerce'
    )

print(f'Raw fuel data: {raw.shape}')
print(raw[['Fecha','Provincia','Producto','PAI','PVP']].head())

In [ ]:
PRODUCT_SLUG = {
    'Gasolina 95 E5':    'Gasolina95',
    'Gasolina 98 E5':    'Gasolina98',
    'Gas\u00f3leo A habitual': 'Gasoleo_A',
    'Gas\u00f3leo Premium':    'Gasoleo_Premium',
}

raw['Producto_slug'] = raw['Producto'].map(PRODUCT_SLUG)
raw = raw.dropna(subset=['Producto_slug'])

monthly_prov = (
    raw.groupby(['Fecha', 'Provincia', 'Producto_slug'])[['PAI', 'PVP']]
    .mean()
    .reset_index()
)

monthly_prov['CCAA'] = monthly_prov['Provincia'].map(PROVINCE_CCAA)
monthly_prov = monthly_prov.dropna(subset=['CCAA'])

monthly_ccaa = (
    monthly_prov.groupby(['Fecha', 'CCAA', 'Producto_slug'])[['PAI', 'PVP']]
    .mean()
    .reset_index()
)

monthly_ccaa = monthly_ccaa.melt(
    id_vars=['Fecha', 'CCAA', 'Producto_slug'],
    value_vars=['PAI', 'PVP'],
    var_name='tipo',
    value_name='precio'
)
monthly_ccaa['col_name'] = monthly_ccaa['tipo'] + '_' + monthly_ccaa['Producto_slug']

precios_wide = monthly_ccaa.pivot_table(
    index=['Fecha', 'CCAA'], columns='col_name', values='precio'
).reset_index()
precios_wide.columns.name = None

print(f'Precios aggregated: {precios_wide.shape}')
print(f'Price columns: {[c for c in precios_wide.columns if c not in ["Fecha","CCAA"]]}')

In [ ]:
master = master.merge(precios_wide, on=['Fecha', 'CCAA'], how='left')

price_cols = [c for c in master.columns if c.startswith('PAI_') or c.startswith('PVP_')]

espana_mask = master['CCAA'] == 'ESPAÑA'
for col in price_cols:
    espana_fill = master.loc[~espana_mask].groupby('Fecha')[col].mean()
    master.loc[espana_mask, col] = master.loc[espana_mask, 'Fecha'].map(espana_fill)

print(f'After precio merge: {master.shape}')
nan_prices = master[price_cols].isna().sum()
print(f'NaN counts per price column:\n{nan_prices}')

## 6 · Final master dataset

In [ ]:
col_order = (
    ['Fecha', 'CCAA', 'Consumo_Tm', 'Target']
    + ['IPI_original', 'IPI_ajustado', 'IPC_var_anual', 'Tasa_paro']
    + ['Precio_Brent_USD']
    + sorted([c for c in price_cols if c.startswith('PVP_')])
    + sorted([c for c in price_cols if c.startswith('PAI_')])
)
master = master[col_order]

print('=== MASTER DATASET ===')
print(f'Shape: {master.shape}')
print(f'Columns ({len(master.columns)}): {list(master.columns)}')
print(f'Date range: {master["Fecha"].min()} → {master["Fecha"].max()}')
print(f'CCAAs: {master["CCAA"].nunique()}')
print(f'Target CCAAs: {master[master["Target"]==1]["CCAA"].unique().tolist()}')
print()
print(master.describe().round(2))

In [ ]:
out_path = DATA_INPUTS / 'master_dataset.csv'
master.to_csv(out_path, index=False)
print(f'Saved → {out_path}  ({master.shape[0]} rows × {master.shape[1]} cols)')

---
## Datasets NOT merged — and why

### 1. `consumo_biodiesel_provincial.csv`
- **Granularity**: Monthly × CCAA × **Province** × **Product type**
- Multiple rows per (Fecha, CCAA) pair due to the Province and Product dimensions.
- Merging would require aggregation (losing provincial/product detail) or create a multi-level index incompatible with a flat master table.
- **Used in**: notebook `01_eda.ipynb` and `02_data_cleaning.ipynb` only.

### 2. `turismo_visitantes_ccaa.csv`
- **Single time point**: only contains data for `2025M10` (October 2025).
- Only covers 3 CCAAs (Andalucía, Cataluña, Madrid) — missing Valencia and the national total.
- Multiple rows per CCAA (breakdowns by access method: Airport, Road, Port, Rail).
- Cannot form a time series or be broadcast to all CCAAs without extensive data acquisition.
- **Recommendation**: download a full historical monthly series from INE Frontur before integrating.